<a href="https://colab.research.google.com/github/Tengoku1/DataScienceProject/blob/main/DataExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
drug_Mortality = pd.read_csv("/content/NCHS_-_Drug_Poisoning_Mortality_by_County__United_States.csv", dtype={"FIPS": str})
drug_Mortality["FIPS"] = drug_Mortality["FIPS"].astype(int) # Convert FIPS to int
drug_Mortality = drug_Mortality[(drug_Mortality["Year"] >= 2019)]
drug_Mortality = drug_Mortality.reset_index(drop=True)
drug_Suseptablity = drug_Mortality.groupby(['FIPS', 'Urban/Rural Category', 'County', 'Population'])['Model-based Death Rate'].mean().reset_index()
drug_Suseptablity["isSuseptible"] = drug_Suseptablity["Model-based Death Rate"] >= drug_Suseptablity["Model-based Death Rate"].mean()

In [5]:
national = pd.read_csv("/content/nationwide-drugs-fy19-fy22_fips_only.csv")
amo = pd.read_csv("/content/amo-drug-seizures-fy19-fy22_fips_only.csv")

In [6]:
amo = amo.dropna(subset="County FIPS")
amo["County FIPS"] = amo['County FIPS'].astype(int)
amo.rename(columns={"Branch" : "Area of Responsibility"}, inplace=True)
amo = amo[~(amo["FY"] == 2022)]

In [7]:
national = national.dropna(subset="County FIPS")
national["County FIPS"] = national['County FIPS'].astype(int)
national = national[~(national["FY"] == 2022)]

In [8]:
df_drug_seizures = pd.concat([national, amo])
df_drug_seizures = df_drug_seizures.reset_index(drop=True)
df_drug_seizures.sort_values("FY", inplace=True)
df_drug_seizures.reset_index(drop=True, inplace=True)
df_drug_seizures["Sum Qty (lbs)"] = df_drug_seizures["Sum Qty (lbs)"].round(2)
df_drug_seizures.rename(columns={"County FIPS": "FIPS"})

,FY,Month (abbv),Component,Region,Land Filter,Area of Responsibility,FIPS,Drug Type,Count of Event,Sum Qty (lbs)
0,2019,APR,Office of Field Operations,Coastal/Interior,Other,CHICAGO FIELD OFFICE,17031,Ketamine,33,0.95
1,2019,APR,Office of Field Operations,Coastal/Interior,Other,CHICAGO FIELD OFFICE,17031,Heroin,16,23.98
2,2019,APR,Office of Field Operations,Coastal/Interior,Other,CHICAGO FIELD OFFICE,17031,Fentanyl,6,1.45
3,2019,APR,Office of Field Operations,Coastal/Interior,Other,CHICAGO FIELD OFFICE,17031,Ecstasy,153,16.12
4,2019,APR,Office of Field Operations,Coastal/Interior,Other,CHICAGO FIELD OFFICE,17031,Cocaine,13,2.66
...,...,...,...,...,...,...,...,...,...,...
8789,2021,SEP,Air and Marine Operations,Coastal/Interior,Other,Houston Air And Marine Branch,48201,Methamphetamine,1,1227.96
8790,2021,SEP,Air and Marine Operations,Coastal/Interior,Other,Houston Air And Marine Branch,48201,Marijuana,1,17.99
8791,2021,SEP,Air and Marine Operations,Coastal/Interior,Other,Houston Air And Marine Branch,48201,Cocaine,1,0.62
8792,2021,OCT,Air and Marine Operations,Southwest Border,Other,San Diego Air And Marine Branch,6073,Methamphetamine,6,2230.34


In [10]:
import requests
import pandas as pd
import time

# 1. Define the Census API Variables we need:
# B19013_001E: Median Household Income
# B17001_001E: Total Population (for poverty calculation)
# B17001_002E: Population below the poverty line
# B23025_003E: Civilian Labor Force (Total)
# B23025_005E: Civilian Labor Force Unemployed
# B15003_001E: Population 25 years and over (Total for education)
# B15003_022E to B15003_025E: Bachelor's, Master's, Professional, and Doctorate degrees

vars_to_pull = "NAME,B19013_001E,B17001_001E,B17001_002E,B23025_003E,B23025_005E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E"

all_years_data = []
years_to_pull = [2019, 2020, 2021]

print("Pulling Extensive Census Data...")
for year in years_to_pull:
    print(f"Fetching {year}...")
    url = f"https://api.census.gov/data/{year}/acs/acs5"

    params = {
        "get": vars_to_pull,
        "for": "county:*",
        "in": "state:*"
    }

    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        df_year = pd.DataFrame(data[1:], columns=data[0])
        df_year['Year'] = year
        all_years_data.append(df_year)
    else:
        print(f"Failed to fetch {year}: {response.status_code}")

    # Rest for 2 full second
    time.sleep(2)

# 2. Combine all three years into one DataFrame
census_combined = pd.concat(all_years_data, ignore_index=True)

# Create FIPS and clean column names
census_combined['FIPS'] = census_combined['state'] + census_combined['county']
census_combined = census_combined.rename(columns={'NAME': 'County_Name'})

# 3. Convert all the raw text columns into numeric values so we can do math
numeric_cols = [
    'B19013_001E', 'B17001_001E', 'B17001_002E',
    'B23025_003E', 'B23025_005E', 'B15003_001E',
    'B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E'
]

for col in numeric_cols:
    census_combined[col] = pd.to_numeric(census_combined[col], errors='coerce')

# ---------------------------------------------------------
# 4. CALCULATE THE ACTUAL PERCENTAGE RATES
# ---------------------------------------------------------
census_combined['Median_Income'] = census_combined['B19013_001E']

# Poverty Rate = (People Below Poverty / Total People) * 100
census_combined['Poverty_Rate (%)'] = (census_combined['B17001_002E'] / census_combined['B17001_001E']) * 100

# Unemployment Rate = (Unemployed / Total Labor Force) * 100
census_combined['Unemployment_Rate (%)'] = (census_combined['B23025_005E'] / census_combined['B23025_003E']) * 100

# Education = (Sum of all higher ed degrees / Total Adults 25+) * 100
census_combined['Higher_Ed_Total'] = (census_combined['B15003_022E'] +
                                      census_combined['B15003_023E'] +
                                      census_combined['B15003_024E'] +
                                      census_combined['B15003_025E'])

census_combined['Bachelors_or_Higher (%)'] = (census_combined['Higher_Ed_Total'] / census_combined['B15003_001E']) * 100

# ---------------------------------------------------------
# 5. KEEP ONLY WHAT WE NEED AND AVERAGE THE 3 YEARS
# ---------------------------------------------------------
final_cols = ['FIPS', 'Year', 'Median_Income', 'Poverty_Rate (%)', 'Unemployment_Rate (%)', 'Bachelors_or_Higher (%)']
census_clean = census_combined[final_cols]

# Group by County and calculate the mean to get the 3-Year Pooled Average
census_3yr_avg = census_clean.groupby(['FIPS']).mean().reset_index()

# Drop the "Year" column since it's an average now, and round the numbers to 2 decimals
census_3yr_avg = census_3yr_avg.drop(columns=['Year'])
census_3yr_avg = census_3yr_avg.round(2)

print("\nSuccess! Here is your Comprehensive 2019-2021 Pooled Census Data:")
print(census_3yr_avg.head())

Pulling Extensive Census Data...
Fetching 2019...
Fetching 2020...
Fetching 2021...

Success! Here is your Comprehensive 2019-2021 Pooled Census Data:
    FIPS  Median_Income  Poverty_Rate (%)  Unemployment_Rate (%)  \
0  01001       59791.00             14.66                   3.14   
1  01003       61474.00              9.58                   3.95   
2  01005       34645.67             28.58                   8.24   
3  01007       51180.00             17.72                   8.14   
4  01009       50370.00             13.51                   4.86   

   Bachelors_or_Higher (%)  
0                    27.67  
1                    32.07  
2                    11.45  
3                    11.21  
4                    13.75  


In [11]:
drug_Suseptablity["FIPS"] = drug_Suseptablity["FIPS"].astype(str)
drug_Suseptablity = drug_Suseptablity.merge(census_3yr_avg, on="FIPS")
drug_Suseptablity["Model-based Death Rate"] = drug_Suseptablity["Model-based Death Rate"].round(2)

In [12]:
drug_Mortality.to_csv("drug_Mortality2019-2021.csv")

In [13]:
df_drug_seizures.to_csv("drug_seizures2019-2021.csv")

In [14]:
drug_Suseptablity.to_csv("df_drug_suseptability2019-2021.csv")